# 02 — Tracking stability

Run detection + ByteTrack on a clip; load the persisted track state and:
- Plot per-track lifespan timelines.
- Count switches/merges per minute.
- Show an example of a track that survives a brief occlusion vs. one that gets a fresh ID after a long gap.

In [ ]:
import sys, pickle
from pathlib import Path
sys.path.insert(0, '..')
import matplotlib.pyplot as plt
from collections import defaultdict

# Run pipeline once with --persist-tracks before opening this notebook:
#   python scripts/run_pipeline.py --input data/raw_clips/match_1080p.mp4 \
#       --start-frame 4500 --max-frames 1500 --no-video --persist-tracks
TRACKS_PKL = Path('../data/outputs/match_1080p_tracks.pkl')
data = pickle.load(open(TRACKS_PKL, 'rb'))
tracked = data['tracked_per_frame']
fps = data['fps']
print(f'{len(tracked)} frames at {fps} fps; {sum(len(t) for t in tracked)} total observations')

In [ ]:
# Per-track lifespan: min/max frame each track_id appears
appearances = defaultdict(list)
for i, frame in enumerate(tracked):
    for tp in frame:
        appearances[tp.track_id].append(i)

tids = sorted(appearances.keys())
fig, ax = plt.subplots(figsize=(12, max(4, len(tids)*0.2)))
for row, tid in enumerate(tids):
    fs = appearances[tid]
    ax.hlines(row, fs[0], fs[-1], colors='steelblue', linewidth=2)
ax.set_yticks(range(len(tids))); ax.set_yticklabels(tids, fontsize=6)
ax.set_xlabel('frame'); ax.set_ylabel('track_id'); ax.set_title('Track lifespans')
plt.tight_layout(); plt.show()

In [ ]:
# Switches/merges per minute: count new track_ids per minute
first_seen = {tid: fs[0] for tid, fs in appearances.items()}
minutes = max(first_seen.values()) / fps / 60 if first_seen else 1
print(f'Tracks created per minute: {len(first_seen) / minutes:.1f}')

In [ ]:
# Distribution of track durations (in seconds) — short ones are likely failed re-acquisitions
durations_s = [(fs[-1] - fs[0] + 1) / fps for fs in appearances.values()]
plt.hist(durations_s, bins=30); plt.xlabel('track duration (s)'); plt.ylabel('count')
plt.title('Track duration distribution'); plt.show()